# Tien schaalkeuzes vast na epoch 50 — voortzetting tot epoch 100

**Onderzoeksvraag:** wat gebeurt er wanneer de gewichten en BN verder leren met de na epoch50 vastgezette schaaloffsets?

We herstellen het **bestaande bevroren checkpoint na epoch55** en voeren uitsluitend **epochs56–100** nieuw uit. De eerste50 epochs zijn gemeenschappelijk; de bevroren pilot51–55 is geërfd. De vergelijking gebruikt de al voltooide doorlerende tienkeuzerun. Er wordt geen nieuwe controletraining uitgevoerd.

De route blijft **E4M3 → E2M3 → E2M1**, met groepen van32, **A32** en **FP32-masters**. Mogelijke factoren: **×1, ×2, ×4, ×8, ×16, ×32, ×64, ×128, ×256 en ×512**.

Deze notebook leest meetbestanden en controleverslagen. Zij laadt geen trainingscheckpoint en start geen training. Alleen SAVE_FIGURES=True schrijft figuren in een nieuwe analysemap; bronresultaten blijven behouden.


In [ ]:
from pathlib import Path
RUN_DIR = Path('.')  # De runkopie bevat hier het exacte resultaatpad.
SAVE_FIGURES = False  # True bewaart alle figuren in een nieuwe analysemap.

In [ ]:
import sys
import importlib.util
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import Markdown, display
except ImportError:
    class Markdown(str):
        pass
    def display(value):
        print(value)

helper_path = RUN_DIR.expanduser().resolve() / 'notebook_support.py'
if not helper_path.is_file():
    raise FileNotFoundError('Stel RUN_DIR in op de resultatenmap van de voortzetting tot epoch 100: ' + str(helper_path.parent))
helper_name = '_mx_freeze_offsets_e50_reference100_analysis'
spec = importlib.util.spec_from_file_location(helper_name, helper_path)
ns = importlib.util.module_from_spec(spec)
sys.modules[helper_name] = ns
spec.loader.exec_module(ns)
run = ns.load_run(RUN_DIR)
figures = {}
display(Markdown(ns.status_markdown(run)))
display(ns.protocol_table(run))
display(ns.diagnostics_table(run))

## Freeze-anker, pilot en nieuwe voortzetting

Voor iedere groep geldt:

$$E_{\mathrm{laag},b}(t)=E_{\mathrm{E4M3},b}(t)+k_b,\qquad k_b\in\{0,\ldots,9\}.$$

De offset blijft de keuze na epoch50; de schaalfactor is $2^{k_b}$. **De E4M3-bronschaal blijft veranderen met de actuele gewichten.** E2M1 deelt de lagere schaal met E2M3. Dit is geen proef met onafhankelijke schalen per exponentbreedte.

De schaal-LR blijft nul. Theta en harde keuzes worden na iedere update gecontroleerd; gradients en Adam-momenten blijven voor deze diagnostische uitvoering berekend. Gewichten en actieve BN-banken leren verder.

### Leeswijze van de accuracygrafieken

Accuracy-assen zoomen in op het gemeten bereik en vermelden dit expliciet. Vergelijkbare precisiepanelen krijgen dezelfde schaal. De drie checkpointfiguren tonen gekoppelde punten op dezelfde ingezoomde accuracy-as; exacte verschillen staan in procentpunt in de tabellen.

Alle meetwaarden blijven zichtbaar. Het volledige verloop bevat ook eventuele lagere beginaccuracies; de afzonderlijke figuur vanaf epoch50 toont het relevante detail in de latere trainingsfase.


In [ ]:
figures['accuracy_after_freeze'] = ns.plot_accuracy_curves(run)
display(figures['accuracy_after_freeze'])
plt.close(figures['accuracy_after_freeze'])

## Primair vergelijkingspunt: epoch 100

We vergelijken de drie precisies op **dezelfde epoch 100**. Dit is het vooraf gekozen hoofdcriterium. De groep hieronder verschijnt pas wanneer epoch 100 werkelijk is gemeten.

Het verschil is **vaste offset minus verder geleerde offset**, uitgedrukt in procentpunt.

In [ ]:
display(ns.checkpoint_table(run, 'endpoint').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
delta = ns.delta_table(run)
if not delta.empty:
    display(delta.loc[delta.Epoch.eq(100)].drop(columns=['weight_bits']).round(6))
figures['accuracy_epoch100'] = ns.plot_checkpoint_accuracy(run, 'endpoint')
display(figures['accuracy_epoch100'])
plt.close(figures['accuracy_epoch100'])

## Aanvullend: E4M3-beste checkpoint na de freeze

Per variant kiezen we het **eerste maximum van de E4M3-testaccuracy binnen epochs 51–100**. Vervolgens tonen we E4M3, E2M3 en E2M1 op precies diezelfde epoch. De twee varianten mogen elk een andere geselecteerde epoch hebben.

Deze selectie beschrijft specifiek de fase na het vastzetten van de offsetkeuzes. Bij een gedeeltelijke run wordt voor beide varianten hetzelfde gemeten deel van 51–100 gebruikt. Selectie op de officiële testset blijft een geselecteerde score en is geen onafhankelijke validatieschatting.

In [ ]:
display(ns.checkpoint_table(run, 'postfreeze').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
figures['accuracy_best_E4_after_freeze'] = ns.plot_checkpoint_accuracy(run, 'postfreeze')
display(figures['accuracy_best_E4_after_freeze'])
plt.close(figures['accuracy_best_E4_after_freeze'])

## Gebruikelijke globale E4M3-selectie over epochs 1–100

We behouden daarnaast het overzicht volgens de gebruikelijke selectie over de **volledige training**. Voor de vaste-offsetvariant bestaat die geschiedenis uit de gemeenschappelijke epochs 1–50 en de bevroren epochs 51–100.

Een globaal beste checkpoint kan uit de **gemeenschappelijke voorfase vóór de freeze** komen. Dat checkpoint wordt expliciet zo gelabeld. De scores ervan beschrijven dan het model van vóór de ingreep en kunnen niet als resultaat van de bevroren voortzetting worden geïnterpreteerd. Alle drie precisies komen ook hier uit één gedeelde E4M3-selectie per variant.

In [ ]:
display(ns.checkpoint_table(run, 'global').drop(columns=['weight_bits']).round({'Testaccuracy [%]': 6}))
figures['accuracy_best_E4_global'] = ns.plot_checkpoint_accuracy(run, 'global')
display(figures['accuracy_best_E4_global'])
plt.close(figures['accuracy_best_E4_global'])
figures['accuracy_full_history'] = ns.plot_accuracy_curves(run, full_history=True)
display(figures['accuracy_full_history'])
plt.close(figures['accuracy_full_history'])

## Accuracyverschillen gedurende de bevroren fase

Voor elke epoch en precisie gebruiken we de bijbehorende meting uit de bestaande dynamische training. De geërfde pilot en de nieuwe voortzetting vormen samen één traject. Opeenvolgende epochs zijn geen onafhankelijke herhalingen.

Positieve verschillen betekenen hogere accuracy met vaste offsets. De tabel bevat de hoofdmeetpunten en de laatste tien gemeten epochs; de figuur toont alle gemeten epochs vanaf het freeze-anker.

In [ ]:
if not delta.empty:
    important_epochs = {51, 55, 75, 76, 90, 91, 100}
    if run.last_epoch is not None:
        important_epochs.update(range(max(51, run.last_epoch - 9), run.last_epoch + 1))
    display(delta.loc[delta.Epoch.isin(important_epochs)].drop(columns=['weight_bits']).round(6))
figures['accuracy_delta'] = ns.plot_accuracy_delta(run)
display(figures['accuracy_delta'])
plt.close(figures['accuracy_delta'])

## Schaalfactorverdeling per geselecteerd checkpoint

De staafdiagrammen tonen **aantal groepen én aandeel** voor het laatste checkpoint, het gemeenschappelijke beste E4M3-checkpoint na freeze en de globale E4M3-selectie. **Alle tien factoren blijven zichtbaar, ook bij nul groepen.** De noemer blijft4.080 groepen; nulwaarden worden niet samengevoegd.

Na freeze moet de verdeling overeenkomen met epoch50. Een globaal geselecteerd checkpoint vóór de freeze kan een andere verdeling hebben: daarvoor wordt de daadwerkelijke schaalstaat van die eerdere epoch gebruikt. De schaalfactorbalken beginnen bij nul; het inzoomen geldt voor accuracyfiguren.


In [ ]:
for role in ('last', 'postfreeze', 'global'):
    display(ns.factor_distribution_table(run, role).round({'Aandeel [%]': 4}))
figures['scale_factor_counts_and_percentages'] = ns.plot_factor_distribution(run)
display(figures['scale_factor_counts_and_percentages'])
plt.close(figures['scale_factor_counts_and_percentages'])


## Controle: individuele keuzes gedurende alle bevroren epochs

Een stabiele totale verdeling volstaat niet: individuele groepen zouden in tegengestelde richting kunnen wisselen. Daarom controleert de runner **iedere FP32-theta en harde offsetkeuze na elke optimizerupdate**. Alle bevroren epochs moeten nul theta-updates en nul offsetwissels tonen.

De historische wisselteller telt gebeurtenissen, geen unieke groepen; één groep kan meermaals binnen een epoch wisselen. De verdelingsfiguren bevatten alle tien factoren op de opgeslagen epochgrenzen.


In [ ]:
freeze = ns.freeze_table(run)
if not freeze.empty:
    display(freeze.loc[freeze.Epoch.isin({51, 55, 75, 76, 90, 91, 100, run.last_epoch})].round(6))
figures['scale_choices_and_switches'] = ns.plot_scale_choices(run)
display(figures['scale_choices_and_switches'])
plt.close(figures['scale_choices_and_switches'])

## Trainingsobjectieven

E4M3 gebruikt harde cross-entropy. E2M3 en E2M1 gebruiken ieder zachte cross-entropy met de vorige precisie als losgekoppelde leraar. Vergelijk de twee trajecten per precisie; de drie trainingsobjectieven zijn onderling verschillend.

In [ ]:
figures['training_objectives'] = ns.plot_objectives(run)
display(figures['training_objectives'])
plt.close(figures['training_objectives'])

## Gemeten duur

De epochtijden omvatten training, evaluatie, controles en checkpointschrijven. **Epochs 51–55 zijn geërfd uit de pilot** en **vanaf epoch 56 is nieuw uitgevoerd**. De samenvatting hieronder houdt die fasen uit elkaar.

Andere uitvoeringsomstandigheden en hervatkosten kunnen de tijd beïnvloeden. Deze FP32-emulatie van MX-gewichten meet geen native MX-versnelling of energieverbruik.

In [ ]:
times = ns.timing_table(run)
if not times.empty:
    display(times.groupby(['Variant', 'Fase'], as_index=False).agg(
        Gemeten_epochs=('Epoch', 'count'),
        Totale_seconden=('Gemeten epochtijd [s]', 'sum'),
        Mediane_epochseconden=('Gemeten epochtijd [s]', 'median')
    ).round(3))
figures['epoch_timing'] = ns.plot_timing(run)
display(figures['epoch_timing'])
plt.close(figures['epoch_timing'])

## Analytische kosten van de schaalkeuze

Dit zijn extra payload- en tensoromvangen, geen gemeten volledige checkpointgrootte, VRAM-piek of geheugentransfer.

- Tien keuzes vereisen **vier bits per groep**:4.080×4/8 = **2.040bytes** selectorinformatie in de flexibele bron. Bij twee keuzes is dat510bytes. Codes0–9 zijn geldig, codes10–15 ongeldig.
- FP32-theta, gradient en twee Adam-momenten omvatten samen **65.280bytes (63,75KiB)**. Andere modeldata, staptellers, tijdelijke buffers en frameworkoverhead zijn niet meegeteld.
- **Theta-LR0 verwijdert deze tensors en berekeningen niet.** Deze diagnostische freeze claimt geen geoptimaliseerde trainingsbesparing.
- Bij één reeds afgeleide lagere precisie wordt de offset verwerkt in de eigen E8M0-schaal. De genoemde selectoren horen bij de flexibele bron waaruit meerdere modi kunnen worden afgeleid.


In [ ]:
display(ns.storage_cost_table(run))
figures['scale_choice_storage_cost'] = ns.plot_storage_cost(run)
display(figures['scale_choice_storage_cost'])
plt.close(figures['scale_choice_storage_cost'])


## Interpretatie en herkomst

- Begin met **epoch 100** en de drie accuracyverschillen tegenover de bestaande dynamische training.
- Bekijk de ontwikkeling over de hele bevroren fase, inclusief de oorspronkelijke LR-overgangen na 75 en 90.
- De E4M3-selectie na de freeze en de globale selectie beantwoorden verschillende vragen. Let bij de globale selectie op de aanduiding vóór/na freeze.
- De geselecteerde scores komen uit de officiële testset. Er is geen afzonderlijke validatieset.
- De bestaande CUDA-instellingen garanderen geen bitexacte herhaling. Eén voortzetting geeft geen schatting van de spreiding tussen runs.
- Deze proef bepaalt niet of epoch 50 het optimale freezemoment is en meet geen geoptimaliseerde besparing van schaaltraining.

De historische dynamische run staat onder `dynamic_reference`; de ongewijzigde pilot onder `pilot_reference`. De notebook verifieert de beschikbare bestandschecksums, de geërfde eerste vijf bevroren epochs, het herstel van checkpoint 55, de voorbeeldvolgorde, de vaste blokkeuzes en de consistentie van JSON/CSV/selecties. Binaire `.pt`-exports worden hier niet opnieuw ingeladen; de bijbehorende controles zijn runtimegerapporteerd.

In [ ]:
if SAVE_FIGURES:
    destination = ns.save_figures(figures, run)
    print('Figuren opgeslagen in:', destination)
else:
    print('Alleen gelezen en weergegeven; bestaande resultaten zijn ongewijzigd.')